# Model 10 - PPO + Qwen2.5-7B + Rich Difficulty Features
### Upgraded from 1.5B to 7B model + 12 new difficulty-detection features
### Expected: higher accuracy ceiling + larger budget std (better per-question allocation)
### Cell 1: Install (skip if done), restart kernel, then Cell 2


In [1]:
# Cell 1: Install
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install',
    'torch', '--index-url', 'https://download.pytorch.org/whl/cu128', '-q'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install',
    'numpy', 'transformers', 'datasets', 'sentence-transformers', 'accelerate', '-q'])
import torch
print('torch:', torch.__version__)
print('cuda:', torch.cuda.is_available())
if torch.cuda.is_available(): print('gpu:', torch.cuda.get_device_name(0))
print('DONE - restart kernel then run Cell 2')

torch: 2.11.0+cu128
cuda: True
gpu: NVIDIA H200 NVL
DONE - restart kernel then run Cell 2


In [2]:
# ============================================================
# Model 10: PPO + 7B Model + Rich Difficulty Features
# Key fixes over Model 8 (39% at 182 tokens, matched fixed budget 184):
#   1. Oracle uses MINIMUM correct budget per question (not highest reward)
#      Easy q solved at 144 -> label 144; hard q needs 210 -> label 210
#      This teaches real per-question efficiency, not "always use 200"
#   2. Full oracle budget range 128-220 restored (need full range for min search)
#   3. Token penalty 0.0005->0.001: stronger efficiency incentive
#   4. entropy_coef 0.01->0.015: budget std collapsed to 3.8, need diversity
#   5. rl_epochs 10->12: was still improving at epoch 10
#   6. All Model 8 fixes retained: ppo_epochs=2, value pretrain, PPO clip, batching
# ============================================================

import os
import re
import json
import copy
import time
import random
import shutil
import hashlib
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer


# ============================================================
# Config
# ============================================================

@dataclass
class Config:
    # Dataset
    train_subset: int = 300
    test_subset: int = 100

    # Models
    llm_name: str = "Qwen/Qwen2.5-7B-Instruct"
    embed_name: str = "sentence-transformers/all-MiniLM-L6-v2"

    # PARALLELISM: how many questions to process at once on GPU
    # H200 150GB VRAM: 7B model ~15GB, batch=8 safe
    llm_batch_size: int = 8

    # Oracle budgets: only 128-220 where model actually works
    # Removed 96 and 112 because model fails 70-74% there
    oracle_budgets: Tuple[int, ...] = (
        128, 144, 160, 176, 192, 210, 220
    )

    # Supervised warm-start
    warmstart_epochs: int = 10
    warmstart_learning_rate: float = 5e-4

    # PPO specific
    rl_epochs: int = 10
    ppo_epochs: int = 2          # FIX: reduced from 4 to 2 - prevents over-updating per rollout
    ppo_epsilon: float = 0.2     # clip range: ratio stays in [0.8, 1.2]
    ppo_batch_size: int = 32     # mini-batch size for PPO updates
    policy_learning_rate: float = 3e-4
    value_learning_rate: float = 8e-4
    entropy_coef: float = 0.02   # higher entropy for budget diversity with 7B
    value_loss_coef: float = 0.5
    grad_clip_norm: float = 1.0

    # Reward: rebalanced penalty so 176-token correct gets positive reward
    # At 176 tokens: 1.0 + 0.25 - 0.0005*176 = 1.162  (correct)
    # At 128 tokens: 1.0 + 0.25 - 0.0005*128 = 1.186  (correct)
    # Small gap encourages efficiency without punishing longer budgets
    reward_token_penalty: float = 0.0005  # 7B more accurate, moderate penalty
    exact_match_bonus: float = 0.25

    # Budget range: min raised to 128 (below this model fails reliably)
    min_budget: int = 96
    max_budget: int = 256

    # Answer generation
    answer_max_new_tokens: int = 20
    do_sample: bool = False
    temperature: float = 0.0

    # RL warmup exploration in 160-220 where model succeeds
    warmup_epochs_rl: int = 2
    warmup_mix_rl: float = 0.25
    warmup_min_budget_high: int = 160

    # Oracle blend in PPO: supervised signal to keep deterministic head moving
    use_oracle_blend: bool = True
    oracle_blend_weight: float = 0.4   # increased: keeps deterministic head learning

    # Hardware
    seed: int = 42
    device: str = "cuda" if torch.cuda.is_available() else "cpu"

    # Logging
    print_every: int = 20
    debug_examples: int = 3
    eval_print_every: int = 10

    # Cache
    cache_dir: str = "./cache_ppo_v15"
    clear_cache_on_start: bool = True
    oracle_cache_path: str = "oracle_labels_v15.json"

    # Saved models
    best_warmstart_path: str = "best_warmstart_v15.pt"
    best_warmstart_acc_path: str = "best_warmstart_acc_v15.pt"
    best_ppo_path: str = "best_ppo_v15.pt"
    best_ppo_acc_path: str = "best_ppo_acc_v15.pt"
    best_value_path: str = "best_value_v15.pt"
    final_policy_path: str = "final_policy_v15.pt"
    final_value_path: str = "final_value_v15.pt"

    # Baselines
    baseline_budgets: Tuple[int, ...] = (96, 128, 160, 192, 224, 256)


CFG = Config()


# ============================================================
# Utilities
# ============================================================

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def ensure_dir(path):
    os.makedirs(path, exist_ok=True)


def reset_cache_dir(cache_dir, retries=5, delay=0.5):
    if not os.path.exists(cache_dir):
        os.makedirs(cache_dir, exist_ok=True)
        return
    last_error = None
    for _ in range(retries):
        try:
            if os.path.exists(cache_dir):
                shutil.rmtree(cache_dir, ignore_errors=False)
            if os.path.exists(cache_dir):
                for root, dirs, files in os.walk(cache_dir, topdown=False):
                    for f in files:
                        try: os.remove(os.path.join(root, f))
                        except FileNotFoundError: pass
                    for d in dirs:
                        try: os.rmdir(os.path.join(root, d))
                        except OSError: pass
                if os.path.exists(cache_dir): os.rmdir(cache_dir)
            break
        except OSError as e:
            last_error = e
            time.sleep(delay)
    if os.path.exists(cache_dir):
        raise RuntimeError(f"Failed to clear cache: {cache_dir}") from last_error
    os.makedirs(cache_dir, exist_ok=True)


def clamp_budget(x, min_budget, max_budget):
    return int(max(min_budget, min(max_budget, round(x))))


def denormalize_budget(x, min_budget, max_budget):
    return x * (max_budget - min_budget) + min_budget


def budget_to_normalized(budget, min_budget, max_budget):
    return float(budget - min_budget) / float(max_budget - min_budget)


def clean_model_output(text):
    if text is None: return ""
    cleaned = text
    for marker in ["\nHuman:", "\nAssistant:", "\nUser:", "\nSystem:",
                   "Human:", "Assistant:", "User:", "System:",
                   "You are an AI assistant", "<|im_end|>", "<|endoftext|>"]:
        if marker in cleaned:
            cleaned = cleaned.split(marker)[0]
    return cleaned.strip()


# ============================================================
# GSM8K helpers
# ============================================================

def extract_gsm8k_final_answer(answer_text):
    if "####" in answer_text:
        return answer_text.split("####")[-1].strip()
    return answer_text.strip()


def normalize_numeric_string(text):
    if text is None: return ""
    text = text.strip().replace(",", "").replace("$", "").replace("%", "").strip()
    matches = re.findall(r"-?\d+(?:\.\d+)?", text)
    if not matches: return text.lower().strip()
    return matches[-1]


def parse_final_answer(output_text):
    text = clean_model_output(output_text)
    for pattern in [
        r"FINAL ANSWER\s*:\s*(-?\d+(?:\.\d+)?)",
        r"Final Answer\s*:\s*(-?\d+(?:\.\d+)?)",
        r"Answer\s*:\s*(-?\d+(?:\.\d+)?)",
    ]:
        m = re.search(pattern, text, re.IGNORECASE)
        if m: return normalize_numeric_string(m.group(1))
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    first_line = lines[0] if lines else text.strip()
    m = re.search(r"-?\d+(?:\.\d+)?", first_line)
    if m: return normalize_numeric_string(m.group(0))
    matches = re.findall(r"-?\d+(?:\.\d+)?", text)
    if matches: return normalize_numeric_string(matches[0])
    return normalize_numeric_string(text)


def is_correct(pred, gold):
    return int(pred == gold)


def compute_reward(pred, gold, thinking_tokens, cfg):
    exact = float(is_correct(pred, gold))
    reward = exact - cfg.reward_token_penalty * float(thinking_tokens)
    if exact > 0.5:
        reward += cfg.exact_match_bonus
    return reward, exact


# ============================================================
# Prompts
# ============================================================

def build_thinking_prompt(question):
    return (
        "Solve the following grade-school math problem carefully.\n"
        "Reason step by step. Keep reasoning concise and relevant.\n"
        "Do not start a conversation. Only produce reasoning.\n\n"
        f"Question: {question}\n\nReasoning:"
    )


def build_answer_prompt(question, thinking_text):
    return (
        "Use the reasoning below to produce the final numeric answer.\n"
        "Return only: FINAL ANSWER: <number>\n\n"
        f"Question: {question}\n\nReasoning:\n{thinking_text}\n\nFINAL ANSWER:"
    )


# ============================================================
# Caching
# ============================================================

def make_cache_key(stage, question, extra, model_name):
    raw = f"{stage}|||{model_name}|||{question}|||{extra}"
    return hashlib.md5(raw.encode()).hexdigest()


def load_cache(cache_dir, key):
    path = os.path.join(cache_dir, f"{key}.json")
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    return None


def save_cache(cache_dir, key, value):
    path = os.path.join(cache_dir, f"{key}.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(value, f, ensure_ascii=False, indent=2)


# ============================================================
# LLM Reasoner with BATCHED GPU inference
# This is the main speedup vs Model 5
# Instead of 1 question at a time, processes batch_size questions
# simultaneously on the GPU - uses the H200 properly
# ============================================================

class LLMReasoner:
    def __init__(self, model_name, device, batch_size=16):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        # LEFT padding is required for decoder-only models in batched mode
        # Without this, the model attends to wrong positions
        self.tokenizer.padding_side = "left"

        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            dtype=torch.float16,
            device_map="auto"
        )
        self.model.eval()
        self.device = device
        self.batch_size = batch_size

        if hasattr(self.model, "generation_config") and self.model.generation_config:
            self.model.generation_config.do_sample = False
            self.model.generation_config.temperature = None
            self.model.generation_config.top_p = None
            self.model.generation_config.top_k = None

    def generate(self, prompt, max_new_tokens, do_sample=False, temperature=0.0):
        """Single generation - keeps backward compatibility with cache system."""
        results = self.generate_batch([prompt], max_new_tokens, do_sample, temperature)
        return results[0]

    def generate_batch(self, prompts, max_new_tokens, do_sample=False, temperature=0.0):
        """
        BATCHED GPU INFERENCE - the key parallelism speedup.

        How it works:
          - Takes a list of prompts
          - Tokenizes them all together with padding
          - GPU processes ALL of them in one forward pass
          - Returns results for all prompts

        Why this is faster:
          - Sequential: GPU runs at 5% utilization (mostly idle waiting)
          - Batched: GPU runs at 70-90% utilization
          - Speedup: 4-8x depending on batch size
        """
        all_results = []

        for i in range(0, len(prompts), self.batch_size):
            batch = prompts[i : i + self.batch_size]

            # Tokenize with padding so all sequences are same length
            inputs = self.tokenizer(
                batch,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=600,
            ).to(self.device)

            input_len = inputs["input_ids"].shape[1]

            gen_kwargs = {
                "max_new_tokens": max_new_tokens,
                "do_sample": do_sample,
                "pad_token_id": self.tokenizer.eos_token_id,
                "eos_token_id": self.tokenizer.eos_token_id,
                "attention_mask": inputs["attention_mask"],
            }
            if do_sample:
                gen_kwargs["temperature"] = temperature

            with torch.no_grad():
                # GPU processes entire batch at once
                outputs = self.model.generate(inputs["input_ids"], **gen_kwargs)

            for output in outputs:
                gen_ids = output[input_len:]
                text = clean_model_output(
                    self.tokenizer.decode(gen_ids, skip_special_tokens=True)
                )
                all_results.append({"text": text, "tokens_used": len(gen_ids)})

        return all_results


# ============================================================
# Two-stage example run (single, uses cache)
# ============================================================

def run_two_stage_example(question, gold_answer, thinking_budget, llm, cfg):
    ensure_dir(cfg.cache_dir)

    # Stage 1: Thinking
    think_key = make_cache_key("think", question, f"budget={thinking_budget}", cfg.llm_name)
    cached = load_cache(cfg.cache_dir, think_key)
    if cached:
        thinking_text = clean_model_output(cached["text"])
        thinking_tokens = cached["tokens_used"]
    else:
        r = llm.generate(build_thinking_prompt(question), thinking_budget)
        thinking_text = clean_model_output(r["text"])
        thinking_tokens = r["tokens_used"]
        save_cache(cfg.cache_dir, think_key, {"text": thinking_text, "tokens_used": thinking_tokens})

    # Stage 2: Answer
    answer_key = make_cache_key("answer", question, thinking_text, cfg.llm_name)
    cached = load_cache(cfg.cache_dir, answer_key)
    if cached:
        answer_text = clean_model_output(cached["text"])
        answer_tokens = cached["tokens_used"]
    else:
        r = llm.generate(build_answer_prompt(question, thinking_text), cfg.answer_max_new_tokens)
        answer_text = clean_model_output(r["text"])
        answer_tokens = r["tokens_used"]
        save_cache(cfg.cache_dir, answer_key, {"text": answer_text, "tokens_used": answer_tokens})

    pred = parse_final_answer(answer_text)
    reward, exact = compute_reward(pred, gold_answer, thinking_tokens, cfg)
    return {
        "prediction": pred, "gold": gold_answer, "correct": int(exact),
        "thinking_tokens_used": thinking_tokens, "answer_tokens_used": answer_tokens,
        "reward": reward, "thinking_text": thinking_text, "answer_text": answer_text,
        "thinking_budget": thinking_budget,
    }


# ============================================================
# BATCHED two-stage run for Oracle and RL rollouts
# Processes multiple questions at once using GPU parallelism
# ============================================================

def run_batch_two_stage(questions, gold_answers, thinking_budget, llm, cfg):
    """
    Run two-stage inference for a BATCH of questions at the same budget.

    Step 1: Check cache for all questions
    Step 2: Run LLM on uncached questions in ONE batched GPU call
    Step 3: Run answer extraction in ONE batched GPU call
    Step 4: Save new results to cache

    This replaces the sequential for-loop that caused the 4-hour runtime.
    """
    ensure_dir(cfg.cache_dir)
    n = len(questions)

    # --- Stage 1: Thinking ---
    thinking_texts = [None] * n
    thinking_tokens_list = [None] * n
    uncached_think_idx = []
    uncached_think_prompts = []

    # Check cache first for all questions
    for i, q in enumerate(questions):
        key = make_cache_key("think", q, f"budget={thinking_budget}", cfg.llm_name)
        cached = load_cache(cfg.cache_dir, key)
        if cached:
            thinking_texts[i] = clean_model_output(cached["text"])
            thinking_tokens_list[i] = cached["tokens_used"]
        else:
            uncached_think_idx.append(i)
            uncached_think_prompts.append(build_thinking_prompt(q))

    # Batch GPU call for all uncached thinking
    if uncached_think_prompts:
        batch_results = llm.generate_batch(uncached_think_prompts, thinking_budget)
        for idx, result in zip(uncached_think_idx, batch_results):
            thinking_texts[idx] = clean_model_output(result["text"])
            thinking_tokens_list[idx] = result["tokens_used"]
            key = make_cache_key("think", questions[idx], f"budget={thinking_budget}", cfg.llm_name)
            save_cache(cfg.cache_dir, key, {
                "text": thinking_texts[idx],
                "tokens_used": thinking_tokens_list[idx]
            })

    # --- Stage 2: Answer extraction ---
    answer_texts = [None] * n
    answer_tokens_list = [None] * n
    uncached_ans_idx = []
    uncached_ans_prompts = []

    for i in range(n):
        key = make_cache_key("answer", questions[i], thinking_texts[i], cfg.llm_name)
        cached = load_cache(cfg.cache_dir, key)
        if cached:
            answer_texts[i] = clean_model_output(cached["text"])
            answer_tokens_list[i] = cached["tokens_used"]
        else:
            uncached_ans_idx.append(i)
            uncached_ans_prompts.append(
                build_answer_prompt(questions[i], thinking_texts[i])
            )

    # Batch GPU call for all uncached answers
    if uncached_ans_prompts:
        batch_results = llm.generate_batch(uncached_ans_prompts, cfg.answer_max_new_tokens)
        for idx, result in zip(uncached_ans_idx, batch_results):
            answer_texts[idx] = clean_model_output(result["text"])
            answer_tokens_list[idx] = result["tokens_used"]
            key = make_cache_key("answer", questions[idx], thinking_texts[idx], cfg.llm_name)
            save_cache(cfg.cache_dir, key, {
                "text": answer_texts[idx],
                "tokens_used": answer_tokens_list[idx]
            })

    # --- Compute rewards ---
    results = []
    for i in range(n):
        pred = parse_final_answer(answer_texts[i])
        reward, exact = compute_reward(pred, gold_answers[i], thinking_tokens_list[i], cfg)
        results.append({
            "prediction": pred, "gold": gold_answers[i], "correct": int(exact),
            "thinking_tokens_used": thinking_tokens_list[i],
            "answer_tokens_used": answer_tokens_list[i],
            "reward": reward, "thinking_text": thinking_texts[i],
            "answer_text": answer_texts[i], "thinking_budget": thinking_budget,
        })
    return results


# ============================================================
# Featurizer
# ============================================================

class QuestionFeaturizer:
    """
    Rich difficulty-aware featurizer.
    New: 12 difficulty features on top of original 50 handcrafted = 62 total handcrafted.
    Total input dim: 384 (SBERT all-MiniLM-L6-v2) + 62 = 446.
    """
    def __init__(self, embed_model_name, device):
        self.embedder = SentenceTransformer(embed_model_name, device=device)

    def handcrafted_features(self, question):
        q = question.lower()
        tokens = q.split()
        nums = re.findall(r"\d+\.?\d*", q)

        # --- Original 50 features (preserved for backward compat) ---
        orig_keywords = [
            "total", "left", "remaining", "each", "every", "more", "less",
            "after", "before", "twice", "half", "times", "altogether",
            "difference", "sum", "product", "cost", "price", "sold",
            "per", "average", "equal", "share", "then", "gave", "bought",
            "earned", "minutes", "hours", "days", "weeks", "fraction",
            "ratio", "percent", "another", "still", "now", "together"
        ]
        orig_feats = [
            len(tokens),
            len(nums),
            len(q),
            max(1, len(re.findall(r"[.!?]", q))),
            1.0 if any(w in q for w in ["more", "sum", "altogether", "total", "together"]) else 0.0,
            1.0 if any(w in q for w in ["left", "remaining", "difference", "less", "still"]) else 0.0,
            1.0 if any(w in q for w in ["each", "every", "times", "product", "twice"]) else 0.0,
            1.0 if any(w in q for w in ["per", "average", "equally", "half", "share", "ratio"]) else 0.0,
            1.0 if any(w in q for w in ["then", "after", "before", "altogether", "remaining", "now"]) else 0.0,
            1.0 if any(w in q for w in ["minutes", "hours", "days", "weeks"]) else 0.0,
            1.0 if "$" in q or any(w in q for w in ["dollar", "dollars", "cost", "price"]) else 0.0,
            1.0 if any(w in q for w in ["half", "fraction", "percent", "ratio"]) else 0.0,
        ] + [1.0 if kw in q else 0.0 for kw in orig_keywords]

        # --- NEW: 12 difficulty-detection features ---

        # 1. Operation count - how many arithmetic operations are implied
        op_words = [
            "add", "added", "plus", "more", "total", "sum", "altogether", "combined",
            "subtract", "subtracted", "minus", "less", "fewer", "remain", "left", "difference",
            "multiply", "multiplied", "times", "product", "double", "triple", "twice",
            "divide", "divided", "split", "share", "each", "per", "half", "quarter",
            "percent", "percentage", "fraction", "ratio", "rate"
        ]
        op_count = float(sum(1 for w in tokens if w in op_words))

        # 2. Numerical value count
        num_count = float(len(nums))

        # 3. Large number flag (>100 implies non-trivial arithmetic)
        has_large_num = 1.0 if any(
            float(n) > 100 for n in nums
            if re.match(r"^\d+\.?\d*$", n)
        ) else 0.0

        # 4. Sentence count - proxy for reasoning steps required
        sentence_count = float(max(1, len(re.findall(r"[.!?]", q)) + 1))

        # 5. Multi-step sequence indicators
        multistep_words = [
            "first", "then", "next", "after that", "finally", "lastly",
            "second", "third", "step", "now", "before", "once"
        ]
        multistep_count = float(sum(1 for w in multistep_words if w in q))

        # 6. Conditional logic (if/when makes questions harder)
        has_conditional = 1.0 if any(w in q for w in [
            "if ", "when ", "unless", "until", "provided"
        ]) else 0.0

        # 7. Comparison operators
        has_comparison = 1.0 if any(w in q for w in [
            "more than", "less than", "greater than", "fewer than",
            "at least", "at most", "how many more", "how many fewer",
            "how much more", "how much less"
        ]) else 0.0

        # 8. Unit conversion (time/distance/weight add steps)
        has_unit_conversion = 1.0 if any(w in q for w in [
            "minute", "hour", "second", "day", "week", "month", "year",
            "inch", "foot", "feet", "yard", "mile", "meter", "kilometer",
            "ounce", "pound", "gram", "kilogram", "liter", "gallon"
        ]) else 0.0

        # 9. Percentage/fraction complexity
        has_pct_fraction = 1.0 if any(w in q for w in [
            "percent", "%", "fraction", "half", "third", "quarter",
            "one-half", "one-third", "two-thirds", "one-quarter"
        ]) else 0.0

        # 10. Entity count (people/objects to track)
        entity_indicators = [
            "he", "she", "they", "his", "her", "their",
            "john", "mary", "tom", "jane", "bob", "alice",
            "sam", "mike", "sarah", "david", "lisa", "james"
        ]
        entity_count = float(sum(1 for w in tokens if w in entity_indicators))

        # 11. Multiple sub-questions
        has_multiple_q = 1.0 if (
            q.count("?") > 1 or "and how" in q or "also how" in q
        ) else 0.0

        # 12. Composite difficulty score (normalized 0-1)
        raw_difficulty = (
            min(op_count, 8.0) / 8.0 * 0.35 +
            min(num_count, 6.0) / 6.0 * 0.25 +
            min(sentence_count, 5.0) / 5.0 * 0.20 +
            min(multistep_count, 4.0) / 4.0 * 0.20
        )

        new_feats = [
            op_count, num_count, has_large_num, sentence_count,
            multistep_count, has_conditional, has_comparison,
            has_unit_conversion, has_pct_fraction, entity_count,
            has_multiple_q, raw_difficulty,
        ]

        return np.array(orig_feats + new_feats, dtype=np.float32)

    def encode(self, question):
        emb = self.embedder.encode(
            question, convert_to_numpy=True, normalize_embeddings=True
        ).astype(np.float32)
        return np.concatenate([emb, self.handcrafted_features(question)], axis=0)

    def encode_batch(self, questions):
        embs = self.embedder.encode(
            questions, convert_to_numpy=True,
            normalize_embeddings=True, batch_size=64
        ).astype(np.float32)
        hand = np.stack([self.handcrafted_features(q) for q in questions])
        return np.concatenate([embs, hand], axis=1)


# ============================================================
# Policy + Value Networks
# ============================================================

class ContinuousBudgetPolicy(nn.Module):
    def __init__(self, input_dim, hidden_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
        )
        self.mean_head = nn.Linear(hidden_dim, 1)
        # log_std = 0.5 gives std ~1.65 for initial exploration
        self.log_std = nn.Parameter(torch.tensor([0.5], dtype=torch.float32))

    def forward(self, x):
        h = self.net(x)
        mean = self.mean_head(h)
        return mean, self.log_std.expand_as(mean)

    def sample_budget(self, x):
        mean, log_std = self.forward(x)
        dist = torch.distributions.Normal(mean, torch.exp(log_std))
        raw = dist.rsample()
        normalized = torch.sigmoid(raw)
        log_prob = dist.log_prob(raw).sum(dim=-1)
        entropy = dist.entropy().sum(dim=-1)
        return normalized, log_prob, entropy

    def get_log_prob(self, x, normalized_budgets):
        """
        Get log probability for given normalized budgets.
        Used in PPO to compute ratio between new and old policy.
        """
        mean, log_std = self.forward(x)
        dist = torch.distributions.Normal(mean, torch.exp(log_std))
        # Inverse sigmoid to get raw values
        raw = torch.logit(normalized_budgets.unsqueeze(-1).clamp(1e-6, 1 - 1e-6))
        log_prob = dist.log_prob(raw).sum(dim=-1)
        entropy = dist.entropy().sum(dim=-1)
        return log_prob, entropy

    def deterministic_budget(self, x):
        mean, _ = self.forward(x)
        return torch.sigmoid(mean)


class ValueNetwork(nn.Module):
    def __init__(self, input_dim, hidden_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


# ============================================================
# Evaluation
# ============================================================

def evaluate_fixed_budget(data, llm, cfg, fixed_budget):
    corrects, tokens, rewards = [], [], []
    questions = [item["question"] for item in data]
    gold_answers = [
        normalize_numeric_string(extract_gsm8k_final_answer(item["answer"]))
        for item in data
    ]
    print(f"\nEvaluating fixed budget = {fixed_budget}...")

    # Process in batches
    for i in range(0, len(questions), cfg.llm_batch_size):
        batch_q = questions[i : i + cfg.llm_batch_size]
        batch_g = gold_answers[i : i + cfg.llm_batch_size]
        results = run_batch_two_stage(batch_q, batch_g, fixed_budget, llm, cfg)
        for r in results:
            corrects.append(r["correct"])
            tokens.append(r["thinking_tokens_used"])
            rewards.append(r["reward"])
        done = min(i + cfg.llm_batch_size, len(questions))
        if done % cfg.eval_print_every == 0 or done == len(questions):
            print(f"  Fixed {fixed_budget} | {done}/{len(questions)} | "
                  f"Acc={np.mean(corrects):.4f} | AvgReward={np.mean(rewards):.4f}")

    return {
        "accuracy": float(np.mean(corrects)),
        "avg_thinking_tokens": float(np.mean(tokens)),
        "avg_reward": float(np.mean(rewards)),
        "budget": fixed_budget
    }


def evaluate_policy_deterministic(data, featurizer, policy, llm, cfg):
    policy.eval()
    corrects, tokens, rewards, budgets = [], [], [], []

    questions = [item["question"] for item in data]
    gold_answers = [
        normalize_numeric_string(extract_gsm8k_final_answer(item["answer"]))
        for item in data
    ]

    # Encode all features at once
    all_feats = featurizer.encode_batch(questions)

    with torch.no_grad():
        feats_tensor = torch.tensor(all_feats, dtype=torch.float32, device=cfg.device)
        norm_budgets = policy.deterministic_budget(feats_tensor).squeeze(-1)
        budget_list = [
            clamp_budget(
                denormalize_budget(nb.item(), cfg.min_budget, cfg.max_budget),
                cfg.min_budget, cfg.max_budget
            )
            for nb in norm_budgets
        ]

    # Group questions by budget for batched inference
    from collections import defaultdict
    budget_groups = defaultdict(list)
    for i, b in enumerate(budget_list):
        budget_groups[b].append(i)

    results_by_idx = {}
    for budget, indices in budget_groups.items():
        batch_q = [questions[i] for i in indices]
        batch_g = [gold_answers[i] for i in indices]
        batch_results = run_batch_two_stage(batch_q, batch_g, budget, llm, cfg)
        for i, r in zip(indices, batch_results):
            results_by_idx[i] = r

    for i in range(len(data)):
        r = results_by_idx[i]
        corrects.append(r["correct"])
        tokens.append(r["thinking_tokens_used"])
        rewards.append(r["reward"])
        budgets.append(budget_list[i])

    if corrects:
        print(f"  Det eval complete | Acc={np.mean(corrects):.4f} | "
              f"AvgBudget={np.mean(budgets):.1f} | StdBudget={np.std(budgets):.1f} | "
              f"Range=[{np.min(budgets):.0f}, {np.max(budgets):.0f}]")

    return {
        "accuracy": float(np.mean(corrects)),
        "avg_thinking_tokens": float(np.mean(tokens)),
        "avg_reward": float(np.mean(rewards)),
        "avg_budget": float(np.mean(budgets)),
        "min_budget": float(np.min(budgets)),
        "max_budget": float(np.max(budgets)),
        "std_budget": float(np.std(budgets)),
    }


# ============================================================
# Oracle builder using BATCHED inference
# Model 5 took ~2.5 hours for this step alone
# With batching: processes 16 questions per budget in one GPU call
# Expected speedup: 3-5x (from ~2.5 hours to ~30-45 min)
# ============================================================

def build_oracle_budget_labels(train_data, llm, cfg):
    if os.path.exists(cfg.oracle_cache_path):
        print(f"\nLoading oracle labels from {cfg.oracle_cache_path}...")
        with open(cfg.oracle_cache_path, "r", encoding="utf-8") as f:
            return json.load(f)

    print("\nBuilding oracle labels with BATCHED GPU inference...")
    questions = [item["question"] for item in train_data]
    gold_answers = [
        normalize_numeric_string(extract_gsm8k_final_answer(item["answer"]))
        for item in train_data
    ]
    n = len(questions)

    # Initialize records
    oracle_records = [
        {"question": q, "gold_answer": g, "budget_rewards": {}, "any_correct": False}
        for q, g in zip(questions, gold_answers)
    ]

    # For each budget, process ALL 300 questions in batches
    # This is much faster than the nested loop in Model 5
    for budget in cfg.oracle_budgets:
        print(f"  Trying budget={budget} for all {n} questions...")
        t0 = time.time()

        for i in range(0, n, cfg.llm_batch_size):
            batch_q = questions[i : i + cfg.llm_batch_size]
            batch_g = gold_answers[i : i + cfg.llm_batch_size]
            batch_results = run_batch_two_stage(batch_q, batch_g, budget, llm, cfg)

            for j, result in enumerate(batch_results):
                idx = i + j
                oracle_records[idx]["budget_rewards"][str(budget)] = {
                    "reward": float(result["reward"]),
                    "correct": int(result["correct"]),
                    "thinking_tokens_used": int(result["thinking_tokens_used"]),
                    "prediction": result["prediction"],
                }
                if result["correct"] == 1:
                    oracle_records[idx]["any_correct"] = True

        elapsed = time.time() - t0
        correct_at_budget = sum(
            1 for r in oracle_records
            if r["budget_rewards"].get(str(budget), {}).get("correct", 0) == 1
        )
        print(f"    Budget {budget} done in {elapsed:.1f}s | "
              f"Correct: {correct_at_budget}/{n} ({100*correct_at_budget/n:.1f}%)")

    # Assign best budget per question
    # FIX from Model 5: use HIGHEST REWARD budget, not minimum correct
    # Model 5 assigned 57% of questions to budget=96 which skewed everything
    for record in oracle_records:
        br = record["budget_rewards"]
        correct_budgets = [int(b) for b, v in br.items() if v["correct"] == 1]
        if correct_budgets:
            best_budget = min(correct_budgets)
        else:
            best_budget = int(max(br.keys(), key=lambda b: br[b]["reward"]))
        record["best_budget"] = best_budget
        record["best_reward"] = float(br[str(best_budget)]["reward"])

    # Print distribution to verify it looks reasonable
    best_budgets = [r["best_budget"] for r in oracle_records]
    solvable = [r["any_correct"] for r in oracle_records]
    print(f"\nOracle stats: mean={np.mean(best_budgets):.1f}, "
          f"std={np.std(best_budgets):.1f}, "
          f"solvable={np.mean(solvable):.3f}")
    for b in sorted(set(best_budgets)):
        count = sum(1 for x in best_budgets if x == b)
        print(f"  budget={b}: {count} questions ({100*count/len(best_budgets):.1f}%)")

    with open(cfg.oracle_cache_path, "w", encoding="utf-8") as f:
        json.dump(oracle_records, f, ensure_ascii=False, indent=2)
    print(f"Saved oracle labels to {cfg.oracle_cache_path}")
    return oracle_records


# ============================================================
# Supervised warm-start
# ============================================================

def warmstart_policy(oracle_records, test_data, featurizer, policy, llm, cfg):
    print("\nStarting supervised warm-start...")
    optimizer = optim.Adam(policy.parameters(), lr=cfg.warmstart_learning_rate)
    mse_loss = nn.MSELoss(reduction="none")

    all_budgets = np.array([r["best_budget"] for r in oracle_records], dtype=np.float32)
    budget_mean = float(all_budgets.mean())
    budget_std = float(all_budgets.std()) + 1e-8

    best_reward = -1e9
    best_accuracy = -1.0

    for epoch in range(cfg.warmstart_epochs):
        policy.train()
        random.shuffle(oracle_records)
        losses, pred_budgets, target_budgets = [], [], []

        for step_idx, record in enumerate(oracle_records):
            target_budget = int(record["best_budget"])
            x = torch.tensor(
                featurizer.encode(record["question"]),
                dtype=torch.float32, device=cfg.device
            ).unsqueeze(0)

            pred_norm = policy.deterministic_budget(x)
            target_norm = torch.tensor(
                [[budget_to_normalized(target_budget, cfg.min_budget, cfg.max_budget)]],
                dtype=torch.float32, device=cfg.device
            )

            # Weight by reward quality + how much this question's budget differs from mean
            reward_weight = max(0.1, float(record["best_reward"]))
            deviation_weight = 1.0 + abs(target_budget - budget_mean) / budget_std
            loss = reward_weight * deviation_weight * mse_loss(pred_norm, target_norm).mean()

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(policy.parameters(), cfg.grad_clip_norm)
            optimizer.step()

            losses.append(float(loss.item()))
            pred_budget = clamp_budget(
                denormalize_budget(pred_norm.item(), cfg.min_budget, cfg.max_budget),
                cfg.min_budget, cfg.max_budget
            )
            pred_budgets.append(pred_budget)
            target_budgets.append(target_budget)

            if (step_idx + 1) % cfg.print_every == 0 or (step_idx + 1) == len(oracle_records):
                mae = np.mean(np.abs(np.array(pred_budgets) - np.array(target_budgets)))
                print(f"[Warmstart {epoch+1}/{cfg.warmstart_epochs}] "
                      f"Step {step_idx+1}/{len(oracle_records)} | "
                      f"Loss={np.mean(losses):.4f} | "
                      f"PredAvg={np.mean(pred_budgets):.1f} | "
                      f"TargetAvg={np.mean(target_budgets):.1f} | MAE={mae:.2f}")

        print(f"Warmstart epoch {epoch+1} done | Loss={np.mean(losses):.4f}")
        eval_det = evaluate_policy_deterministic(test_data, featurizer, policy, llm, cfg)

        if eval_det["avg_reward"] > best_reward:
            best_reward = eval_det["avg_reward"]
            torch.save(policy.state_dict(), cfg.best_warmstart_path)
            print(f"  Saved best warmstart by reward = {best_reward:.4f}")

        if eval_det["accuracy"] > best_accuracy:
            best_accuracy = eval_det["accuracy"]
            torch.save(policy.state_dict(), cfg.best_warmstart_acc_path)
            print(f"  Saved best warmstart by accuracy = {best_accuracy:.4f}")


# ============================================================
# PPO Training
# This is the core fix over Model 5's basic policy gradient
#
# How PPO works here:
#   1. ROLLOUT: collect 300 (question, budget, reward) experiences
#              using current policy - all done with batched GPU calls
#   2. UPDATE:  run 4 epochs of mini-batch updates using the PPO clip
#              The clip prevents any single update from being too large
#   3. REPEAT:  for each RL epoch
#
# Why clip stops the oscillation:
#   - Old code: loss = -(log_prob * advantage) → unbounded update size
#   - PPO:      ratio = exp(new_log_prob - old_log_prob)
#               loss = -min(ratio*A, clip(ratio,0.8,1.2)*A)
#               If ratio > 1.2 or < 0.8, gradient is zeroed out
#               Policy cannot change by more than 20% per update
# ============================================================

def train_ppo(train_data, test_data, featurizer, policy, value_net,
              llm, cfg, oracle_records=None):
    print("\nStarting PPO training...")
    print(f"  PPO epsilon (clip range): {cfg.ppo_epsilon}")
    print(f"  PPO epochs per rollout: {cfg.ppo_epochs}")
    print(f"  Batch size: {cfg.llm_batch_size} questions at once on GPU")

    # Oracle lookup for blend signal
    oracle_lookup = {}
    if cfg.use_oracle_blend and oracle_records:
        for rec in oracle_records:
            oracle_lookup[rec["question"]] = rec["best_budget"]

    policy_optimizer = optim.Adam(policy.parameters(), lr=cfg.policy_learning_rate)
    value_optimizer = optim.Adam(value_net.parameters(), lr=cfg.value_learning_rate)
    mse_loss_fn = nn.MSELoss()

    questions = [item["question"] for item in train_data]
    gold_answers = [
        normalize_numeric_string(extract_gsm8k_final_answer(item["answer"]))
        for item in train_data
    ]

    # Pre-encode ALL training features once (saves time every epoch)
    print("  Pre-encoding all training features...")
    all_feats = featurizer.encode_batch(questions)
    all_feats_tensor = torch.tensor(all_feats, dtype=torch.float32, device=CFG.device)

    best_test_reward = -1e9
    best_test_accuracy = -1.0

    for epoch in range(cfg.rl_epochs):
        print(f"\n{'='*80}")
        print(f"PPO Epoch {epoch+1}/{cfg.rl_epochs}")
        print(f"{'='*80}")

        # --------------------------------------------------------
        # PHASE 1: ROLLOUT - collect experiences
        # Sample budgets for all questions, run LLM, get rewards
        # All LLM calls are batched for GPU parallelism
        # --------------------------------------------------------
        print("  Phase 1: Collecting rollout experiences (batched)...")
        policy.eval()

        rollout_idx = list(range(len(questions)))
        random.shuffle(rollout_idx)

        # Sample budgets for ALL questions at once
        with torch.no_grad():
            norm_budgets, old_log_probs, _ = policy.sample_budget(all_feats_tensor)
            norm_budgets = norm_budgets.squeeze(-1)
            old_log_probs = old_log_probs.detach()

        # Convert to actual budgets
        budget_list = []
        for i in range(len(questions)):
            nb = norm_budgets[i].item()
            b = clamp_budget(
                denormalize_budget(nb, cfg.min_budget, cfg.max_budget),
                cfg.min_budget, cfg.max_budget
            )
            # Warmup: mix in high budgets during first epochs
            if epoch < cfg.warmup_epochs_rl and random.random() < cfg.warmup_mix_rl:
                b = random.randint(cfg.warmup_min_budget_high, cfg.max_budget)
            budget_list.append(b)

        # Group questions by budget, run batched LLM inference
        from collections import defaultdict
        budget_groups = defaultdict(list)
        for i, b in enumerate(budget_list):
            budget_groups[b].append(i)

        rewards_list = [None] * len(questions)
        corrects_list = [None] * len(questions)
        tokens_list = [None] * len(questions)

        for budget, indices in budget_groups.items():
            batch_q = [questions[i] for i in indices]
            batch_g = [gold_answers[i] for i in indices]
            batch_results = run_batch_two_stage(batch_q, batch_g, budget, llm, cfg)
            for i, r in zip(indices, batch_results):
                rewards_list[i] = float(r["reward"])
                corrects_list[i] = int(r["correct"])
                tokens_list[i] = int(r["thinking_tokens_used"])

        rewards_tensor = torch.tensor(rewards_list, dtype=torch.float32, device=cfg.device)

        # Compute advantages
        with torch.no_grad():
            values = value_net(all_feats_tensor)
            advantages = rewards_tensor - values
            # Normalize advantages for training stability
            advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

        print(f"  Rollout done | "
              f"Acc={np.mean(corrects_list):.4f} | "
              f"AvgReward={np.mean(rewards_list):.4f} | "
              f"AvgBudget={np.mean(budget_list):.1f} | "
              f"StdBudget={np.std(budget_list):.1f}")

        # --------------------------------------------------------
        # PHASE 2: PPO UPDATE - update policy using clip
        # Run ppo_epochs times on the same rollout data
        # Mini-batches shuffled each time for better gradient estimates
        # --------------------------------------------------------
        print(f"  Phase 2: PPO updates ({cfg.ppo_epochs} epochs, batch={cfg.ppo_batch_size})...")
        policy.train()

        norm_budgets_stored = norm_budgets.detach()

        for ppo_ep in range(cfg.ppo_epochs):
            indices = torch.randperm(len(questions))
            ep_policy_losses = []
            ep_value_losses = []
            ep_clip_fracs = []

            for start in range(0, len(questions), cfg.ppo_batch_size):
                batch_idx = indices[start : start + cfg.ppo_batch_size]

                batch_feats = all_feats_tensor[batch_idx]
                batch_old_log_probs = old_log_probs[batch_idx]
                batch_advantages = advantages[batch_idx]
                batch_rewards = rewards_tensor[batch_idx]
                batch_norm_budgets = norm_budgets_stored[batch_idx]

                # Get NEW log probs from updated policy
                new_log_probs, new_entropy = policy.get_log_prob(
                    batch_feats, batch_norm_budgets
                )

                # PPO ratio: how much has the policy changed?
                ratio = torch.exp(new_log_probs - batch_old_log_probs)

                # THE PPO CLIP - this is the fix for the oscillation
                # ratio > 1.2: policy is going too far in a good direction, cap it
                # ratio < 0.8: policy is going too far in a bad direction, cap it
                clipped_ratio = torch.clamp(
                    ratio,
                    1.0 - cfg.ppo_epsilon,
                    1.0 + cfg.ppo_epsilon
                )

                # Take the MINIMUM of clipped and unclipped
                # This is the pessimistic bound that prevents over-updating
                policy_loss = -torch.min(
                    ratio * batch_advantages,
                    clipped_ratio * batch_advantages
                ).mean()

                # Entropy bonus: keep policy from collapsing to one budget
                entropy_loss = -cfg.entropy_coef * new_entropy.mean()

                # Oracle blend: supervised signal on deterministic head
                # Keeps the deterministic head learning even during PPO
                supervised_loss = torch.tensor(0.0, device=cfg.device)
                if cfg.use_oracle_blend:
                    oracle_targets = []
                    valid_idx = []
                    for k, bi in enumerate(batch_idx.tolist()):
                        q = questions[bi]
                        if q in oracle_lookup:
                            oracle_targets.append(
                                budget_to_normalized(
                                    oracle_lookup[q], cfg.min_budget, cfg.max_budget
                                )
                            )
                            valid_idx.append(k)

                    if oracle_targets:
                        valid_feats = batch_feats[valid_idx]
                        oracle_norm_t = torch.tensor(
                            oracle_targets, dtype=torch.float32, device=cfg.device
                        ).unsqueeze(1)
                        det_pred = policy.deterministic_budget(valid_feats)
                        supervised_loss = nn.functional.mse_loss(det_pred, oracle_norm_t)

                total_policy_loss = (
                    policy_loss + entropy_loss + cfg.oracle_blend_weight * supervised_loss
                )

                policy_optimizer.zero_grad()
                total_policy_loss.backward()
                torch.nn.utils.clip_grad_norm_(policy.parameters(), cfg.grad_clip_norm)
                policy_optimizer.step()

                # Value network update
                value_preds = value_net(batch_feats)
                value_loss = mse_loss_fn(value_preds, batch_rewards)
                value_optimizer.zero_grad()
                (cfg.value_loss_coef * value_loss).backward()
                torch.nn.utils.clip_grad_norm_(value_net.parameters(), cfg.grad_clip_norm)
                value_optimizer.step()

                # Track clip fraction (diagnostic: how often does clip activate?)
                clip_frac = ((ratio - 1.0).abs() > cfg.ppo_epsilon).float().mean().item()
                ep_policy_losses.append(policy_loss.item())
                ep_value_losses.append(value_loss.item())
                ep_clip_fracs.append(clip_frac)

            print(f"    PPO update epoch {ppo_ep+1}/{cfg.ppo_epochs} | "
                  f"PolicyLoss={np.mean(ep_policy_losses):.4f} | "
                  f"ValueLoss={np.mean(ep_value_losses):.4f} | "
                  f"ClipFrac={np.mean(ep_clip_fracs):.3f}")
            # ClipFrac guide:
            #   ~0.0: updates too small, increase LR or decrease epsilon
            #   0.1-0.3: healthy range
            #   >0.5: updates too large, something is wrong

        # Evaluate after each RL epoch
        print("\n  Evaluating deterministic policy on test set...")
        eval_det = evaluate_policy_deterministic(test_data, featurizer, policy, llm, cfg)
        print(f"  Test Accuracy  : {eval_det['accuracy']:.4f}")
        print(f"  Test AvgTokens : {eval_det['avg_thinking_tokens']:.1f}")
        print(f"  Test AvgReward : {eval_det['avg_reward']:.4f}")
        print(f"  Test Budget    : {eval_det['avg_budget']:.1f} +/- "
              f"{eval_det['std_budget']:.1f} "
              f"[{eval_det['min_budget']:.0f}, {eval_det['max_budget']:.0f}]")

        if eval_det["avg_reward"] > best_test_reward:
            best_test_reward = eval_det["avg_reward"]
            torch.save(policy.state_dict(), cfg.best_ppo_path)
            torch.save(value_net.state_dict(), cfg.best_value_path)
            print(f"  Saved best PPO model by reward = {best_test_reward:.4f}")

        if eval_det["accuracy"] > best_test_accuracy:
            best_test_accuracy = eval_det["accuracy"]
            torch.save(policy.state_dict(), cfg.best_ppo_acc_path)
            print(f"  Saved best PPO model by accuracy = {best_test_accuracy:.4f}")


# ============================================================
# Main
# ============================================================

def main():
    set_seed(CFG.seed)

    print("torch version:", torch.__version__)
    print("cuda available:", torch.cuda.is_available())
    print("cuda device count:", torch.cuda.device_count())
    if torch.cuda.is_available():
        print("gpu name:", torch.cuda.get_device_name(0))
        print("gpu memory:", torch.cuda.get_device_properties(0).total_memory // 1e9, "GB")
    else:
        print("WARNING: No GPU detected. This will be very slow.")

    if CFG.clear_cache_on_start:
        print(f"\nClearing cache: {CFG.cache_dir}")
        reset_cache_dir(CFG.cache_dir)
        # Also remove any stale oracle cache from previous runs
        # This was the Model 6 bug: it loaded wrong oracle labels silently
        for stale in [CFG.oracle_cache_path,
                      "oracle_labels_v11.json", "oracle_labels_v12.json", "oracle_labels_v13.json"]:
            if os.path.exists(stale):
                os.remove(stale)
                print(f"  Removed stale oracle cache: {stale}")
    else:
        ensure_dir(CFG.cache_dir)

    print(f"Using device: {CFG.device}")
    print(f"LLM batch size: {CFG.llm_batch_size} (questions processed in parallel)")

    print("\nLoading GSM8K...")
    ds = load_dataset("gsm8k", "main")
    train_data = list(ds["train"].select(range(min(CFG.train_subset, len(ds["train"])))))
    test_data = list(ds["test"].select(range(min(CFG.test_subset, len(ds["test"])))))
    print(f"Train: {len(train_data)} | Test: {len(test_data)}")

    print("\nLoading featurizer...")
    featurizer = QuestionFeaturizer(CFG.embed_name, CFG.device)
    input_dim = featurizer.encode(train_data[0]["question"]).shape[0]
    print(f"Feature dim: {input_dim}")

    print("\nLoading policy and value network...")
    policy = ContinuousBudgetPolicy(input_dim=input_dim).to(CFG.device)
    value_net = ValueNetwork(input_dim=input_dim).to(CFG.device)

    # Initialize toward 176 tokens (normalized ~0.72 for range 128-220)
    with torch.no_grad():
        policy.mean_head.bias.fill_(0.85)

    print("\nLoading LLM (with batched inference)...")
    llm = LLMReasoner(CFG.llm_name, CFG.device, batch_size=CFG.llm_batch_size)

    # Fixed budget baselines
    print("\n" + "="*70)
    print("FIXED BUDGET BASELINES")
    print("="*70)
    baseline_results = {}
    for b in CFG.baseline_budgets:
        res = evaluate_fixed_budget(test_data, llm, CFG, b)
        baseline_results[b] = res
        print(f"Fixed {b:>3} | Acc={res['accuracy']:.4f} | "
              f"AvgTokens={res['avg_thinking_tokens']:.0f} | "
              f"AvgReward={res['avg_reward']:.4f}")

    # Build oracle labels
    oracle_records = build_oracle_budget_labels(train_data, llm, CFG)

    # Warm-start
    warmstart_policy(oracle_records, test_data, featurizer, policy, llm, CFG)

    # Load best warm-start
    if os.path.exists(CFG.best_warmstart_acc_path):
        print("\nLoading best warm-start policy by accuracy...")
        policy.load_state_dict(torch.load(CFG.best_warmstart_acc_path, map_location=CFG.device))
    elif os.path.exists(CFG.best_warmstart_path):
        print("\nLoading best warm-start policy by reward...")
        policy.load_state_dict(torch.load(CFG.best_warmstart_path, map_location=CFG.device))

    print("\nPost warm-start evaluation:")
    warm_eval = evaluate_policy_deterministic(test_data, featurizer, policy, llm, CFG)
    print(json.dumps({"warmstart_eval": warm_eval}, indent=2))

    # Pre-compute all training features (used for value pre-training and PPO)
    print("\nPre-computing all training features...")
    all_feats = featurizer.encode_batch([item["question"] for item in train_data])
    all_feats_tensor = torch.tensor(all_feats, dtype=torch.float32, device=CFG.device)

    # --------------------------------------------------------
    # FIX: Pre-train value network before PPO starts
    # Without this, ValueLoss starts at 2.6 (random init)
    # causing garbage advantage estimates in PPO epoch 1
    # Pre-training brings it down so epoch 1 updates are clean
    # --------------------------------------------------------
    print("\nPre-training value network on oracle rewards...")
    value_pretrain_optimizer = optim.Adam(value_net.parameters(), lr=CFG.value_learning_rate)
    oracle_avg_reward = float(np.mean([r["best_reward"] for r in oracle_records]))
    print(f"  Target oracle avg reward: {oracle_avg_reward:.4f}")

    value_net.train()
    for pretrain_ep in range(5):
        # Shuffle for better gradient estimates
        idx_perm = torch.randperm(len(train_data))
        ep_losses = []
        for start in range(0, len(train_data), 32):
            batch_idx = idx_perm[start:start+32]
            batch_feats = all_feats_tensor[batch_idx]

            # Per-question target: use oracle reward if available, else avg
            batch_targets = []
            for bi in batch_idx.tolist():
                q = train_data[bi]["question"]
                rec = next((r for r in oracle_records if r["question"] == q), None)
                target = float(rec["best_reward"]) if rec else oracle_avg_reward
                batch_targets.append(target)

            target_t = torch.tensor(batch_targets, dtype=torch.float32, device=CFG.device)
            preds = value_net(batch_feats)
            loss = nn.functional.mse_loss(preds, target_t)
            value_pretrain_optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(value_net.parameters(), CFG.grad_clip_norm)
            value_pretrain_optimizer.step()
            ep_losses.append(loss.item())

        print(f"  Value pretrain epoch {pretrain_ep+1}/5 | Loss={np.mean(ep_losses):.4f}")

    print("Value pre-training complete. PPO will start with better advantage estimates.\n")

    # PPO Training
    train_ppo(
        train_data=train_data,
        test_data=test_data,
        featurizer=featurizer,
        policy=policy,
        value_net=value_net,
        llm=llm,
        cfg=CFG,
        oracle_records=oracle_records,
    )

    # Load best PPO model
    print("\nLoading best PPO model for final evaluation...")
    if os.path.exists(CFG.best_ppo_acc_path):
        policy.load_state_dict(torch.load(CFG.best_ppo_acc_path, map_location=CFG.device))
    elif os.path.exists(CFG.best_ppo_path):
        policy.load_state_dict(torch.load(CFG.best_ppo_path, map_location=CFG.device))
    if os.path.exists(CFG.best_value_path):
        value_net.load_state_dict(torch.load(CFG.best_value_path, map_location=CFG.device))

    print("\nFinal deterministic policy evaluation:")
    final_det = evaluate_policy_deterministic(test_data, featurizer, policy, llm, CFG)
    print(json.dumps({"final_deterministic_eval": final_det}, indent=2))

    torch.save(policy.state_dict(), CFG.final_policy_path)
    torch.save(value_net.state_dict(), CFG.final_value_path)
    print(f"\nSaved: {CFG.final_policy_path}")
    print(f"Saved: {CFG.final_value_path}")

    # Final comparison table
    print("\n" + "="*72)
    print("FINAL RESULTS COMPARISON")
    print("="*72)
    print(f"{'Method':<35} {'Acc':>6} {'AvgTokens':>10} {'AvgReward':>10}")
    print("-"*72)
    for b, res in baseline_results.items():
        print(f"  Fixed budget {b:<22} {res['accuracy']:>6.4f} "
              f"{res['avg_thinking_tokens']:>10.0f} {res['avg_reward']:>10.4f}")
    print(f"  Policy (PPO deterministic)     {final_det['accuracy']:>6.4f} "
          f"{final_det['avg_thinking_tokens']:>10.0f} {final_det['avg_reward']:>10.4f}")
    print("="*72)
    print("\nAI Disclosure: Written with Claude's assistance")


main()

/opt/conda/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch version: 2.11.0+cu128
cuda available: True
cuda device count: 1
gpu name: NVIDIA H200 NVL
gpu memory: 150.0 GB

Clearing cache: ./cache_ppo_v15
Using device: cuda
LLM batch size: 8 (questions processed in parallel)

Loading GSM8K...


Train: 300 | Test: 100

Loading featurizer...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12236.95it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Feature dim: 446

Loading policy and value network...

Loading LLM (with batched inference)...


Loading weights: 100%|██████████| 339/339 [00:39<00:00,  8.52it/s]



FIXED BUDGET BASELINES

Evaluating fixed budget = 96...
  Fixed 96 | 40/100 | Acc=0.4750 | AvgReward=0.5457
  Fixed 96 | 80/100 | Acc=0.4625 | AvgReward=0.5301
  Fixed 96 | 100/100 | Acc=0.4700 | AvgReward=0.5395
Fixed  96 | Acc=0.4700 | AvgTokens=96 | AvgReward=0.5395

Evaluating fixed budget = 128...
  Fixed 128 | 40/100 | Acc=0.5500 | AvgReward=0.6235
  Fixed 128 | 80/100 | Acc=0.6250 | AvgReward=0.7172
  Fixed 128 | 100/100 | Acc=0.6300 | AvgReward=0.7235
Fixed 128 | Acc=0.6300 | AvgTokens=128 | AvgReward=0.7235

Evaluating fixed budget = 160...
  Fixed 160 | 40/100 | Acc=0.6750 | AvgReward=0.7637
  Fixed 160 | 80/100 | Acc=0.7125 | AvgReward=0.8106
  Fixed 160 | 100/100 | Acc=0.7400 | AvgReward=0.8450
Fixed 160 | Acc=0.7400 | AvgTokens=160 | AvgReward=0.8450

Evaluating fixed budget = 192...
  Fixed 192 | 40/100 | Acc=0.6750 | AvgReward=0.7477
  Fixed 192 | 80/100 | Acc=0.7375 | AvgReward=0.8259
  Fixed 192 | 100/100 | Acc=0.7600 | AvgReward=0.8540
Fixed 192 | Acc=0.7600 | AvgTok